In [2]:
!pip install -U \
  transformers \
  datasets \
  torchaudio \
  librosa \
  accelerate \
  evaluate \
  scikit-learn \
  soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/

In [1]:
import torch
import torchaudio
import transformers
import datasets

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
Device: Tesla T4


In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
from datasets import load_dataset

languages = {
    "Tamil": "ta",
    "Malayalam": "ml",
    "Telugu": "te",
    "Kannada": "kn"
}

datasets_per_lang = {}

for lang_name, lang_code in languages.items():
    print(f"Streaming {lang_name}...")
    ds = load_dataset(
        "ai4bharat/indicvoices_r",
        lang_name,
        split="train",
        streaming=True
    )
    datasets_per_lang[lang_code] = ds

Streaming Tamil...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/137 [00:00<?, ?it/s]

Streaming Malayalam...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/185 [00:00<?, ?it/s]

Streaming Kannada...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

In [17]:
!pip install --upgrade torch torchvision torchaudio

  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse_cu12-12.5.8.93-py3-none-ma

In [2]:
import torchcodec
from collections import defaultdict

real_samples = []
counts = defaultdict(int)

for lang_code, ds in datasets_per_lang.items():
    for ex in ds:
        if counts[lang_code] >= 50:
            break
        real_samples.append({
            "audio": ex["audio"],
            "label": 0,
            "language": lang_code
        })
        counts[lang_code] += 1

counts, len(real_samples)

(defaultdict(int, {'ta': 50, 'ml': 50, 'te': 50, 'kn': 50}), 200)

In [3]:
from datasets import Dataset

real_ds = Dataset.from_list(real_samples)
real_ds

Dataset({
    features: ['audio', 'label', 'language'],
    num_rows: 200
})

In [4]:
!pip install TTS #not working

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

In [4]:
from datasets import load_dataset

ai_stream = load_dataset(
    "Bisher/ASVspoof_2019_LA",
    split="train",
    streaming=True
)

In [5]:
for ex in ai_stream:
    print(ex)
    break

{'speaker_id': 'LA_0079', 'audio_file_name': 'LA_T_1138215', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7ad3a22048c0>, 'system_id': '-', 'key': 0}


In [6]:
ai_samples = []

for ex in ai_stream:
    if ex["key"] == 1:   # SPOOF = AI
        ai_samples.append({
            "audio": ex["audio"],
            "label": 1,          # AI
            "language": "unk"    # unknown / mixed
        })
    if len(ai_samples) >= 200:
        break

len(ai_samples)

200

In [7]:
from datasets import Dataset

ai_ds = Dataset.from_list(ai_samples)
ai_ds

Dataset({
    features: ['audio', 'label', 'language'],
    num_rows: 200
})

In [8]:
from datasets import concatenate_datasets

full_ds = concatenate_datasets([real_ds, ai_ds])
full_ds

Dataset({
    features: ['audio', 'label', 'language'],
    num_rows: 400
})

In [9]:
from transformers import AutoFeatureExtractor

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)


In [10]:
def preprocess(batch):
    audio = batch["audio"]

    inputs = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        max_length=16000 * 5,  # 5 seconds
        truncation=True,
    )

    batch["input_values"] = inputs["input_values"][0]
    return batch


In [11]:
from datasets import Audio

full_ds = full_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

In [12]:
full_ds = full_ds.map(
    preprocess,
    remove_columns=["audio"],
    desc="Extracting XLS-R features"
)

Extracting XLS-R features:   0%|          | 0/400 [00:00<?, ? examples/s]

In [13]:
full_ds

Dataset({
    features: ['label', 'language', 'input_values'],
    num_rows: 400
})

In [14]:
len(full_ds[0]["input_values"])

80000

In [15]:
full_ds = full_ds.train_test_split(test_size=0.2, seed=42)

train_ds = full_ds["train"]
eval_ds = full_ds["test"]

len(train_ds), len(eval_ds)

(320, 80)

In [30]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 752.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 124.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 67.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 115.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.1 MB/s eta 0:00:00

In [2]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is

In [16]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)

NumPy: 1.26.4
Torch: 2.10.0+cu128


In [17]:
from transformers import AutoModelForAudioClassification

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    num_labels=2,
    use_safetensors=True,   # 🔐 bypass torch.load restriction
)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
model

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./xlsr-ai-vs-human",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=5,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=20,
    save_total_limit=2,
    report_to="none"
)


In [20]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def compute_metrics(pred):
    logits = pred.predictions
    labels = pred.label_ids

    # probability/logit for class "AI"
    ai_scores = logits[:, 1]

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, ai_scores)
    }


In [25]:
import torch
from typing import List, Dict

class DataCollatorAudioWithPadding:
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Extract input_values and labels
        input_values = [torch.tensor(f["input_values"]) for f in features]
        labels = torch.tensor([f["label"] for f in features])

        # Pad input_values
        input_values_padded = torch.nn.utils.rnn.pad_sequence(
            input_values,
            batch_first=True,
            padding_value=0.0
        )

        # Create attention mask (1 = real, 0 = padding)
        attention_mask = torch.zeros_like(input_values_padded)
        for i, seq in enumerate(input_values):
            attention_mask[i, : seq.shape[0]] = 1

        return {
            "input_values": input_values_padded,
            "attention_mask": attention_mask,
            "labels": labels
        }


In [26]:
data_collator = DataCollatorAudioWithPadding()

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,   # ✅ THIS IS THE FIX
    compute_metrics=compute_metrics
)

In [27]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Auc
50,1.292230,0.535861,1.000000,1.000000
100,0.679576,0.237370,1.000000,1.000000
150,0.470529,0.138084,1.000000,1.000000
200,0.346912,0.113374,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.7553971862792969, metrics={'train_runtime': 350.768, 'train_samples_per_second': 4.561, 'train_steps_per_second': 0.57, 'total_flos': 2.3875632635335254e+17, 'train_loss': 0.7553971862792969, 'epoch': 5.0})

In [28]:
trainer.evaluate()

{'eval_loss': 0.11337433010339737,
 'eval_accuracy': 1.0,
 'eval_auc': 1.0,
 'eval_runtime': 8.3658,
 'eval_samples_per_second': 9.563,
 'eval_steps_per_second': 2.391,
 'epoch': 5.0}

In [30]:
from datasets import load_dataset

ai_eval_stream = load_dataset(
    "Bisher/ASVspoof_2019_LA",
    split="validation",
    streaming=True
)

In [31]:
eval_ai_samples = []

for ex in ai_eval_stream:
    if ex["key"] == 1:  # spoof
        eval_ai_samples.append({
            "audio": ex["audio"],
            "label": 1,
            "language": "unk"
        })
    if len(eval_ai_samples) >= 100:
        break

len(eval_ai_samples)


100

In [32]:
from datasets import Dataset, Audio

eval_ai_ds = Dataset.from_list(eval_ai_samples)

eval_ai_ds = eval_ai_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

eval_ai_ds = eval_ai_ds.map(
    preprocess,
    remove_columns=["audio"]
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [33]:
trainer.evaluate(eval_ai_ds)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


{'eval_loss': 0.09761718660593033,
 'eval_accuracy': 1.0,
 'eval_auc': nan,
 'eval_runtime': 4.4408,
 'eval_samples_per_second': 22.519,
 'eval_steps_per_second': 5.63,
 'epoch': 5.0}

In [35]:
from datasets import load_dataset

real_eval_stream = load_dataset(
    "ai4bharat/indicvoices_r",
    "Tamil",           # pick ONE language for clarity
    split="test",
    streaming=True
)

Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/137 [00:00<?, ?it/s]

In [36]:
real_eval_samples = []

for ex in real_eval_stream:
    real_eval_samples.append({
        "audio": ex["audio"],
        "label": 0,     # HUMAN
        "language": "ta"
    })
    if len(real_eval_samples) >= 100:
        break

len(real_eval_samples)

100

In [37]:
from datasets import concatenate_datasets, Dataset

balanced_eval_ds = Dataset.from_list(real_eval_samples + eval_ai_samples)

In [38]:
balanced_eval_ds = balanced_eval_ds.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

balanced_eval_ds = balanced_eval_ds.map(
    preprocess,
    remove_columns=["audio"]
)

trainer.evaluate(balanced_eval_ds)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'eval_loss': 0.11509338021278381,
 'eval_accuracy': 1.0,
 'eval_auc': 1.0,
 'eval_runtime': 11.3089,
 'eval_samples_per_second': 17.685,
 'eval_steps_per_second': 4.421,
 'epoch': 5.0}

In [39]:
for param in model.wav2vec2.parameters():
    param.requires_grad = False


In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [41]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Auc
50,0.192297,0.050324,1.000000,1.000000
100,0.104716,0.028600,1.000000,1.000000
150,0.093174,0.020693,1.000000,1.000000
200,0.114053,0.018666,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.1353744888305664, metrics={'train_runtime': 180.9803, 'train_samples_per_second': 8.841, 'train_steps_per_second': 1.105, 'total_flos': 2.3875632635335254e+17, 'train_loss': 0.1353744888305664, 'epoch': 5.0})

In [42]:
trainer.evaluate(balanced_eval_ds)

{'eval_loss': 0.018978308886289597,
 'eval_accuracy': 1.0,
 'eval_auc': 1.0,
 'eval_runtime': 11.2605,
 'eval_samples_per_second': 17.761,
 'eval_steps_per_second': 4.44,
 'epoch': 5.0}

In [43]:
import numpy as np

def add_noise(audio, noise_level=0.005):
    """
    audio: np.ndarray
    noise_level: small value = subtle noise
    """
    noise = np.random.randn(len(audio))
    return audio + noise_level * noise


In [44]:
def preprocess(batch):
    audio = batch["audio"]["array"]
    sampling_rate = batch["audio"]["sampling_rate"]

    # 👇 add noise ONLY to human speech
    if batch["label"] == 0:
        audio = add_noise(audio, noise_level=0.005)

    inputs = feature_extractor(
        audio,
        sampling_rate=sampling_rate,
        max_length=16000 * 5,
        truncation=True,
    )

    batch["input_values"] = inputs["input_values"][0]
    return batch

In [46]:
full_ds_raw = concatenate_datasets([real_ds, ai_ds])

In [47]:
from datasets import Audio

full_ds_raw = full_ds_raw.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

In [48]:
def preprocess_with_noise(batch):
    audio = batch["audio"]["array"]
    sampling_rate = batch["audio"]["sampling_rate"]

    # Add noise ONLY to human speech
    if batch["label"] == 0:
        audio = add_noise(audio, noise_level=0.005)

    inputs = feature_extractor(
        audio,
        sampling_rate=sampling_rate,
        max_length=16000 * 5,
        truncation=True,
    )

    batch["input_values"] = inputs["input_values"][0]
    return batch

In [49]:
full_ds = full_ds_raw.map(
    preprocess_with_noise,
    remove_columns=["audio"],
    desc="Preprocessing with noisy human speech"
)

Preprocessing with noisy human speech:   0%|          | 0/400 [00:00<?, ? examples/s]

In [50]:
full_ds = full_ds.train_test_split(test_size=0.2, seed=42)

train_ds = full_ds["train"]
eval_ds = full_ds["test"]

In [51]:
for param in model.wav2vec2.parameters():
    param.requires_grad = False

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


Step,Training Loss,Validation Loss,Accuracy,Auc
50,0.039542,0.007522,1.000000,1.000000
100,0.027246,0.004603,1.000000,1.000000
150,0.027324,0.003522,1.000000,1.000000
200,0.068739,0.003211,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.04114247798919678, metrics={'train_runtime': 186.7856, 'train_samples_per_second': 8.566, 'train_steps_per_second': 1.071, 'total_flos': 2.3875632635335254e+17, 'train_loss': 0.04114247798919678, 'epoch': 5.0})

In [53]:
trainer.evaluate(balanced_eval_ds)

{'eval_loss': 0.003236799268051982,
 'eval_accuracy': 1.0,
 'eval_auc': 1.0,
 'eval_runtime': 11.146,
 'eval_samples_per_second': 17.944,
 'eval_steps_per_second': 4.486,
 'epoch': 5.0}

In [54]:
SAVE_DIR = "./ai_vs_human_xlsr"

trainer.save_model(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./ai_vs_human_xlsr/preprocessor_config.json']

In [55]:
import torch
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForAudioClassification.from_pretrained(SAVE_DIR).to(device)
feature_extractor = AutoFeatureExtractor.from_pretrained(SAVE_DIR)

model.eval()

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [56]:
import librosa
import numpy as np

def predict_audio(path):
    # Load audio
    audio, sr = librosa.load(path, sr=16000)

    # Extract features
    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        truncation=True,
        max_length=16000 * 5
    )

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    ai_prob = probs[0, 1].item()
    human_prob = probs[0, 0].item()

    return {
        "human_probability": round(human_prob, 4),
        "ai_probability": round(ai_prob, 4),
        "prediction": "AI" if ai_prob > 0.5 else "Human"
    }

In [57]:
predict_audio("/content/AUD-20250417-WA0012.wav")

{'human_probability': 0.9964, 'ai_probability': 0.0036, 'prediction': 'Human'}

In [60]:
predict_audio("/content/samplevoice1.wav")

{'human_probability': 0.9952, 'ai_probability': 0.0048, 'prediction': 'Human'}